# Indexing a material on a substrate (1b/3)

Index and refine the grains of a material whose Laue patterns also contain the spots of a substrate (single crystal or a few grains). The substrate spots must not be indexed as (or mixed with) spots of the material. Two strategies (`STRATEGY` in SETTINGS):

| STRATEGY | in each image | when |
|---|---|---|
| `'refine'` | the substrate orientation (found on a reference image) is refined first, its spots are removed, then the grains of the material are indexed on the remaining spots | substrate spots move from image to image (bending, drift, depth...); gives also a map of the substrate orientation (drift check) |
| `'blacklist'` | the spots of the substrate in the reference image are removed (same pixel positions within `BLACKLIST_TOLERANCE`), then the material is indexed | substrate spots at fixed positions on the detector (faster) |

Results (folder `RESULTS_ROOT`): `fitfiles_substrate/<prefix>####_g0.fit` (substrate, `'refine'`) and `fitfiles_<material>/<prefix>####_g0.fit, _g1.fit ...` (material). A spot of a .cor file belongs to one grain only; `spot_index` in .fit files is the row of the .cor file.

The indexation is a **sequence of steps** (`sequence` section of the configuration, module `LaueTools.indexing_sequence`): here substrate then material. For several materials: `1c_index_refine_multimaterials.ipynb`.

**This notebook is part of the LaueTools package** (https://github.com/BM32ESRF/lauetools). Tested with python 3.12

| notebook | input | output |
|---|---|---|
| `1_index_refine.ipynb` | `.cor` files | `.fit` files of one material |
| **`1b_index_refine_substrate.ipynb`** | `.cor` files | `.fit` files of the substrate and of the material |
| `1c_index_refine_multimaterials.ipynb` | `.cor` files | `.fit` files of several materials |
| `2_postprocess_maps.ipynb`, `3_segmentation_grains.ipynb` | `.fit` files of one folder | maps, grains |

# SETTINGS

In [ ]:
CONFIG = 'configs/322812_Al_Al2Cu.yaml'          # configuration of the experiment (folders, material of the sample, indexing)
NEW_CONFIG = 'configs/322812_Al_on_Al2O3.yaml'   # configuration written by this notebook (with the sequence of indexing)
RESULTS_ROOT = None    # folder of the fitfiles_<...> folders. None: <parent of fit_folder of CONFIG>/sequence_<name of NEW_CONFIG>

STRATEGY = 'refine'    # 'refine' or 'blacklist' (see above)

# substrate
SUBSTRATE = 'Al2O3'                 # material of the substrate
SUBSTRATE_UB = 'Al2O3_1'            # name of its orientation matrix in ub_matrices of CONFIG (tested first). None: indexed from scratch
REFERENCE_IMAGE = None              # image to find the substrate orientation (None: run: test_image_index of CONFIG)
NB_SUBSTRATE_GRAINS = 1
SUBSTRATE_INDEXING = {'central spots indices': 'to9', 'MAXINDEXPROBED': 30, 'nlutmax': 4}   # changes of indexing: dict_indexrefine
SUBSTRATE_ACCEPT = {'min_nb_indexed': 20, 'min_matching_rate': 20}      # substrate grain kept if fulfilled
BLACKLIST_TOLERANCE = 2.            # pixel ('blacklist' strategy)

# material of the sample (key_material of CONFIG)
NB_GRAINS = 3                       # max nb of grains of the material per image
SAMPLE_INDEXING = {'central spots indices': 'to9', 'MAXINDEXPROBED': 25, 'nlutmax': 4}
SAMPLE_ACCEPT = {'min_nb_indexed': 10, 'min_matching_rate': 15, 'max_pixdev': 1.5}
MIN_GAIN = 8                        # a grain must index at least MIN_GAIN spots (else: end of indexing of the image)

IMAGE_RANGE = None     # [first, last] image indices of the batch. None: selection of CONFIG (all files by default)
NB_CPUS = None         # None: nb_cpus of CONFIG (or all cpus)
RUN_BATCH = True
EXECUTION = 'local'    # 'local' or 'slurm'
SAVE_RESULTS = True

# IMPORTS

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"   # before numpy import: 1 thread per process (images are dispatched on cpus)
import copy
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import ipympl  # interactive plots (zoom, hover) in JupyterLab / VS Code
    %matplotlib widget
except ImportError:
    %matplotlib inline

import LaueTools.indexing_batch as IB
import LaueTools.indexing_sequence as IS
import LaueTools.map_workflow as MW
import LaueTools.IOLaueTools as IOLT
import LaueTools.generaltools as GT

print('LaueTools from', Path(IB.__file__).parent)
print(f'nb of cpus available: {os.cpu_count()}')

# LOAD CONFIGURATION

In [ ]:
cfg = IB.load_config(CONFIG)
IB.print_config(cfg)
IB.check_config(cfg)
key_material = cfg['material']['key_material']
mapdims = cfg['scan']['mapdims']
ref_image = cfg['run']['test_image_index'] if REFERENCE_IMAGE is None else REFERENCE_IMAGE
results_root = (Path(RESULTS_ROOT) if RESULTS_ROOT
                else Path(cfg['output']['fit_folder']).parent / f'sequence_{Path(NEW_CONFIG).stem}')
print(f'\nmaterial: {key_material} | substrate: {SUBSTRATE} | strategy: {STRATEGY}\nresults in {results_root}')

# 1. SUBSTRATE orientation on the reference image

The substrate is indexed on `REFERENCE_IMAGE` (with `SUBSTRATE_UB` tested first if given). Its refined orientation is stored as `substrate_ref` in `ub_matrices` of the new configuration; its .fit file (folder `reference_substrate`, not a result folder) is the blacklist of the `'blacklist'` strategy.

If the substrate is not found: change `SUBSTRATE_INDEXING` (e.g. more central spots, larger `MAXINDEXPROBED`, `nlutmax`) or `REFERENCE_IMAGE`.

In [ ]:
refcor = IB.corfile_path(cfg, ref_image)
ub_matrices = dict(cfg['ub_matrices'] or {})
known = (IB.get_previousUB({'run': {'usepreviousUB': [SUBSTRATE_UB]}, 'ub_matrices': ub_matrices})
         if SUBSTRATE_UB in ub_matrices else False)
params_sub = IB.params_from_config(cfg, build_LUT=False, key_material=SUBSTRATE, nbGrainstoFind=NB_SUBSTRATE_GRAINS,
                                   usepreviousUB=known, skipindexing=False, useinternalmultiprocessing=True,
                                   outputlistindices=True, ignorefitfileresults=True, blacklist=None,
                                   fit_folder=str(results_root / 'reference_substrate'))
params_sub.dict_indexrefine.update(SUBSTRATE_INDEXING)
params_sub.build_LUT()

res_sub = IB.index_refine(refcor, params_sub)
IB.print_result(res_sub, minnbindexed=SUBSTRATE_ACCEPT.get('min_nb_indexed', 20),
                minMR=SUBSTRATE_ACCEPT.get('min_matching_rate', 20))
substrate_found = bool(res_sub[1]) and res_sub[1][0][1] is not None
if substrate_found:
    ub_matrices['substrate_ref'] = np.asarray(res_sub[1][0][1], dtype=float).tolist()
    substrate_fitfile = params_sub.fitfile_folder(ref_image) / f'{refcor.stem}_g0.fit'
    GT.printgreen(f'substrate orientation stored as substrate_ref; .fit file: {substrate_fitfile}')
else:
    GT.printred('substrate not found on the reference image: change SUBSTRATE_INDEXING or REFERENCE_IMAGE')

In [ ]:
# experimental spots (grey) and substrate spots (red) of the reference image
data = IOLT.readfile_cor(str(refcor))[0]
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(data[:, 2], data[:, 3], s=8, c='0.6', label=f'{len(data)} spots')
if substrate_found:
    xy = data[res_sub[-1], 2:4]
    ax.scatter(xy[:, 0], xy[:, 1], s=40, facecolors='none', edgecolors='r', label=f'substrate: {len(xy)} spots')
ax.invert_yaxis(); ax.legend(fontsize=8); ax.set_title(refcor.name, fontsize=9)
plt.show()

# 2. SEQUENCE of indexing → new configuration file

- `'refine'`: step `substrate` (orientation `substrate_ref` refined, or indexed from scratch if it does not fit), then step `sample` (the material) on the remaining spots
- `'blacklist'`: `indexing: blacklist` = substrate spots of the reference image, then step `sample`

The sequence is written in `NEW_CONFIG` (section `sequence`): it can also be edited there (documentation: `LaueTools/indexing_sequence.py`).

In [ ]:
sample_step = {'name': 'sample', 'key_material': key_material, 'nbGrainstoFind': NB_GRAINS,
               'dict_indexrefine': SAMPLE_INDEXING, 'accept': SAMPLE_ACCEPT}
changes = {'blacklist': None}
if STRATEGY == 'refine':
    steps = [{'name': 'substrate', 'key_material': SUBSTRATE, 'folder': 'substrate', 'nbGrainstoFind': NB_SUBSTRATE_GRAINS,
              'usepreviousUB': ['substrate_ref'], 'skipindexing': False, 'dict_indexrefine': SUBSTRATE_INDEXING,
              'accept': SUBSTRATE_ACCEPT},
             sample_step]
elif STRATEGY == 'blacklist':
    steps = [sample_step]
    changes.update(blacklist=str(substrate_fitfile), blacklist_tolerance=BLACKLIST_TOLERANCE)
else:
    raise ValueError("STRATEGY must be 'refine' or 'blacklist'")
if IMAGE_RANGE is not None:
    changes['image_range'] = list(IMAGE_RANGE)

sequence = {'strategy': 'ordered', 'min_gain_indexed_spots': MIN_GAIN, 'folders_root': str(results_root), 'steps': steps}
cfg_seq = IB.save_config(cfg, NEW_CONFIG, overwrite=True, sequence=sequence, ub_matrices=ub_matrices,
                         fit_folder=str(results_root), results_folder=str(results_root), results_stamp='sequence',
                         **changes)
config_ok = IB.check_config(cfg_seq)

# 3. TEST on 1 image

All the attempts of the sequence are printed (kept grains, grains not found or rejected by the `accept` criteria, gain too small). The .fit files of this image are written in the result folders.

In [ ]:
TEST_IMAGE = ref_image
testcor = IB.corfile_path(cfg_seq, TEST_IMAGE)
params_test = IB.params_from_config(cfg_seq, useinternalmultiprocessing=True, ignorefitfileresults=True)
t0 = time.time()
res = IB.index_refine(testcor, params_test)
IS.print_records(res)
print(f'elapsed time {time.time() - t0:.1f} s')
IS.plot_records(res, testcor)
plt.show()

In [ ]:
# [OPTION] a few successive images
TEST_SEVERAL = False
if TEST_SEVERAL:
    for imageindex in range(TEST_IMAGE, TEST_IMAGE + 3):
        IS.print_records(IB.index_refine(IB.corfile_path(cfg_seq, imageindex), params_test), trace=False)

# BATCH: all .cor files of the selection

- set of files: section `selection` of the new configuration (`IMAGE_RANGE` in SETTINGS, or all files)
- `EXECUTION = 'slurm'` (SETTINGS): job(s) on the ESRF cluster with exactly these parameters (cells below)
- run again: the .fit files of the analysed images are replaced (`run: ignorefitfileresults: false` in the configuration: only images without results)

In [ ]:
params = IB.params_from_config(cfg_seq)
listfiles = IB.select_corfiles(cfg_seq)
print(f'{len(listfiles)} .cor files to analyse, from {Path(listfiles[0]).name} to {Path(listfiles[-1]).name}')
nb_cpus = NB_CPUS or cfg_seq['run']['nb_cpus'] or IB.available_cpus()

allresults = None
if RUN_BATCH and config_ok and EXECUTION == 'local':
    t0 = time.time()
    allresults = IB.run_multiprocessing(listfiles, params, nb_cpus=nb_cpus)
    GT.printgreen(f'Done in {time.time() - t0:.0f} s')

## [OPTION] BATCH on SLURM (ESRF cluster)

With `EXECUTION = 'slurm'` in SETTINGS: the job folder (parameters, script, logs, results) is written in `slurm_jobs/` next to the results. Submit from this notebook in a jupyter-slurm session (`SUBMIT = True`), or with the printed `sbatch` command.

In [ ]:
SLURM_MACHINE = 'magnifix'   # 'auto', 'magnifix', 'magnifix2', 'magnifix3', 'hpc6', 'hpc7', 'hpc8', 'nice'
SLURM_TIME = '1:00:00'       # time limit hh:mm:ss
SLURM_CPUS = 192             # cpus per job (1 node); magnifix: always 192 (whole node, ESRF IT)
NCHUNKS = 1                  # nb of jobs (job array)
SUBMIT = False               # True: sbatch now (jupyter-slurm session)

job = None
if EXECUTION == 'slurm' and config_ok:
    machine, ncpus = IB.choose_slurm_machine(SLURM_CPUS) if SLURM_MACHINE == 'auto' else (SLURM_MACHINE, SLURM_CPUS)
    job = IB.prepare_slurm_job(cfg_seq, params, listfiles, machine=machine, nb_cpus=ncpus, time=SLURM_TIME,
                               nchunks=NCHUNKS, jobs_folder=results_root / 'slurm_jobs')
    if SUBMIT:
        IB.submit_slurm_job(job)

In [ ]:
# job state (run again to update), then results of the finished job(s)
if EXECUTION == 'slurm' and job is not None:
    IB.slurm_job_status(job)
    allresults = IB.load_slurm_results(job)

## SUMMARY of the batch

- nb of indexed spots of the first grain of each folder, nb of grains of the material
- `'refine'`: misorientation of the substrate to its reference orientation (no symmetry): drift, bending, or wrong indexation

In [ ]:
if allresults is None:
    raise RuntimeError('no batch results: set RUN_BATCH = True (and check the configuration)')
maps = IS.summarize_sequence(allresults, mapdims)
folders = [k for k in maps if k != 'nb_grains_total']
panels = [(maps[f]['nb_indexed_g0'], f'fitfiles_{f}: nb indexed spots (grain 0)') for f in folders]
if key_material in maps:
    panels.append((maps[key_material]['nb_grains'], f'{key_material}: nb of grains'))
if STRATEGY == 'refine' and 'substrate' in maps:
    panels.append((IS.misorientation_map(allresults, mapdims, 'substrate', ub_matrices['substrate_ref']),
                   'substrate: misorientation to reference (deg)'))
fig, axes = plt.subplots(1, len(panels), figsize=(4.6 * len(panels), 4.2), squeeze=False)
for ax, (data, title) in zip(axes[0], panels):
    MW.plot_map(data, cfg_seq, title, ax=ax)
fig.tight_layout()
plt.show()

## SAVE results and configurations of each folder

- `allresults_sequence.pickle` in the results folder (reload: `IB.load_allresults(path)`, kept grains of image `res`: `IS.records_of(res)`)
- one configuration per folder of .fit files (`<NEW_CONFIG>_<folder>.yaml`: its material and `fit_folder`, no sequence) to use as `CONFIG` in `2_postprocess_maps.ipynb` and `3_segmentation_grains.ipynb`

In [ ]:
if SAVE_RESULTS and allresults is not None:
    IB.save_allresults(cfg_seq, params, allresults, IB.summarize_results(allresults, mapdims), overwrite=True,
                       nb_cpus=nb_cpus)
    for folder in IS.step_folders(params.sequence):
        cfg_folder = IS.folder_config(cfg_seq, folder)
        print(f"fitfiles_{folder}: CONFIG = '{cfg_folder['configfile']}'")

**Next step:** `2_postprocess_maps.ipynb` with `CONFIG` = the configuration of the folder of the material (printed above)

## Export of this notebook (HTML, PDF)

Save the notebook first (Ctrl+S): the export reads the saved `.ipynb` file, with its outputs. Files `<notebook name>.html` / `.pdf`. Same as in a terminal: `lauetools-export-notebook <notebook>.ipynb`

In [ ]:
from LaueTools.notebook_export import export_this_notebook

EXPORT_TO = ['html', 'pdf']
EXPORT_FOLDER = None    # None: folder of the notebook
export_this_notebook(to=EXPORT_TO, outdir=EXPORT_FOLDER);